In [4]:
from fdgen.paths import TABLES_DIR
import pandas as pd

path = TABLES_DIR / "car_sales.csv"

# Load with a forgiving decoder in case of bad chars
df = pd.read_csv(path, encoding="utf-8", encoding_errors="replace")

# Normalize for comparison
df["__model_norm"] = df["Model"].str.strip().str.lower()
df["__company_norm"] = df["Company"].str.strip().str.lower()

# Find models mapping to multiple companies
violating_models = (
    df.groupby("__model_norm")["__company_norm"]
      .nunique()
      .reset_index(name="company_count")
      .query("company_count > 1")
)

print(f"Total models: {df['__model_norm'].nunique()}")
print(f"Models violating Model -> Company: {len(violating_models)}\n")

if violating_models.empty:
    print("No violations found.")
else:
    # List the conflicting pairs
    conflicts = (
        df.loc[df["__model_norm"].isin(violating_models["__model_norm"])]
          .groupby("__model_norm")["__company_norm"]
          .unique()
          .reset_index(name="companies")
    )
    print("Conflicting model→companies:")
    for _, row in conflicts.iterrows():
        print(f"- {row['__model_norm']}: {', '.join(sorted(row['companies']))}")

    # Show the actual offending rows for inspection
    offenders = df[df["__model_norm"].isin(violating_models["__model_norm"])]
    cols_to_show = ["Car_id", "Model", "Company", "Dealer_Name", "Price ($)", "Color"]
    print("\nSample offending rows:")
    print(offenders[cols_to_show].sort_values("Model").head(50).to_string(index=False))


Total models: 154
Models violating Model -> Company: 1

Conflicting model→companies:
- neon: dodge, plymouth

Sample offending rows:
      Car_id Model  Company                                     Dealer_Name  Price ($)      Color
C_CND_000365  Neon Plymouth               Scrivener Performance Engineering      22500 Pale White
C_CND_015471  Neon Plymouth                         Clay Johnson Auto Sales      29001      Black
C_CND_015632  Neon    Dodge                          Chrysler of Tri-Cities      19000 Pale White
C_CND_015637  Neon Plymouth                                       U-Haul CO       9600      Black
C_CND_015737  Neon Plymouth                            Star Enterprises Inc      29000      Black
C_CND_015826  Neon    Dodge                                     Capitol KIA      19000 Pale White
C_CND_016019  Neon Plymouth                  Ryder Truck Rental and Leasing      27000 Pale White
C_CND_016029  Neon Plymouth                     Motor Vehicle Branch Office      29

In [6]:
def report_fd_violations(path, lhs_cols, rhs_col, show_rows=50):
    df = pd.read_csv(path, encoding="utf-8", encoding_errors="replace")

    # normalize for comparison
    def norm(col):
        return df[col].astype(str).str.strip().str.lower()
    lhs_norm = [f"{c}__norm" for c in lhs_cols]
    rhs_norm = f"{rhs_col}__norm"
    for c, nc in zip(lhs_cols, lhs_norm):
        df[nc] = norm(c)
    df[rhs_norm] = norm(rhs_col)

    # find violating keys (LHS -> multiple RHS)
    violating_keys = (
        df.groupby(lhs_norm)[rhs_norm]
          .nunique()
          .reset_index(name="rhs_count")
          .query("rhs_count > 1")
    )

    print(f"Total {lhs_cols[-1].lower()}s: {df[lhs_norm[0]].nunique()}")  # tailored for single-col LHS
    print(f"{lhs_cols[-1]}s violating {' + '.join(lhs_cols)} -> {rhs_col}: {len(violating_keys)}\n")

    if violating_keys.empty:
        print("No violations found.")
        return

    # list conflicting pairs
    conflicts = (
        df[df[lhs_norm[0]].isin(violating_keys[lhs_norm[0]])]
          .groupby(lhs_norm[0])[rhs_norm]
          .unique()
          .reset_index(name="rhs_values")
    )
    print(f"Conflicting {lhs_cols[-1]}→{rhs_col}:")
    for _, row in conflicts.iterrows():
        lhs_val = row[lhs_norm[0]]
        rhs_vals = ", ".join(sorted(row["rhs_values"]))
        print(f"- {lhs_val}: {rhs_vals}")

    # show offending rows
    offenders = df[df[lhs_norm[0]].isin(violating_keys[lhs_norm[0]])]
    cols_to_show = ["Car_id", "Model", "Company", "Dealer_Name", "Price ($)", "Color"]
    print("\nSample offending rows:")
    print(offenders[cols_to_show].sort_values("Model").head(show_rows).to_string(index=False))

In [ ]:
path = TABLES_DIR / "car_sales.csv"
report_fd_violations(path, lhs_cols=["Mode"], rhs_col="Dealer_Region", show_rows=50)

Total dealer_names: 28
Dealer_Names violating Dealer_Name -> Dealer_Region: 28

Conflicting Dealer_Name→Dealer_Region:
- buddy storbeck's diesel service inc: aurora, austin, greenville, janesville, middletown, pasco, scottsdale
- c & m motors inc: aurora, austin, greenville, janesville, middletown, pasco, scottsdale
- capitol kia: aurora, austin, greenville, janesville, middletown, pasco, scottsdale
- chrysler of tri-cities: aurora, austin, greenville, janesville, middletown, pasco, scottsdale
- chrysler plymouth: aurora, austin, greenville, janesville, middletown, pasco, scottsdale
- classic chevy: aurora, austin, greenville, janesville, middletown, pasco, scottsdale
- clay johnson auto sales: aurora, austin, greenville, janesville, middletown, pasco, scottsdale
- diehl motor co inc: aurora, austin, greenville, janesville, middletown, pasco, scottsdale
- enterprise rent a car: aurora, austin, greenville, janesville, middletown, pasco, scottsdale
- gartner buick hyundai saab: aurora, a